# 🕳️ Seeing what's missing

Every real dataset has holes. The dangerous mistake is treating them as a nuisance to delete — because **the pattern of what's missing is itself data.** Who refused to answer, and why, can matter more than the answers you received.

In this lesson you will use **`missingno`**, a library built for exactly one job: *making missing data visible*. Then you will prove that our survey's gaps are not random — and measure how badly they distort the findings.

First, install it. It is not built into the browser's Python, so we add it — this takes a few seconds.

In [ ]:
%pip install missingno

import missingno as msno
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/eu_media_habits.csv")
print("Loaded", len(df), "survey responses")

The blunt summary first — how many values are missing in each column?

In [ ]:
df.isna().sum()

Four columns have gaps. But *counts* tell you nothing about **pattern** — and pattern is where the story is.

## The missingness matrix

Every row of the dataset is a thin horizontal line. **White gaps = missing values.**

In [ ]:
msno.matrix(df)
plt.show()

Look at the two right-hand columns with gaps (`trust_news`, `pays_for_news`) and at `household_income_band`. If missingness were purely random, the white marks would be scattered like static. Do some gaps appear to line up on the same rows?

## Completeness per column

In [ ]:
msno.bar(df)
plt.show()

## The three flavours of missing

Statisticians distinguish three kinds, and the difference decides what you are allowed to do about them:

| Type | Meaning | Example | Is it safe? |
|---|---|---|---|
| **MCAR** | Missing Completely At Random | a sensor glitched | Mostly harmless |
| **MAR** | Missing At Random *given other columns* | older people skip a question | Fixable if you model it |
| **MNAR** | Missing **because of the answer itself** | people who distrust the media refuse the trust question | **Dangerous — cannot be fixed from the data** |

You cannot read these off a chart directly, but you *can* gather strong evidence. Let's investigate.

## Do gaps travel together?

The heatmap shows **correlation between missingness** — not between values. +1 means: when one is missing, the other almost always is too.

In [ ]:
msno.heatmap(df)
plt.show()

There it is: **`household_income_band` and `pays_for_news` co-disappear at about +0.75.** That is not a coincidence — it is a *person*. Some respondents guard their financial privacy, so they skip both money-related questions.

Meanwhile `hours_tv`'s gaps correlate with nothing (≈0) — a hint that it is MCAR, the harmless kind.

The dendrogram groups columns that go missing together:

In [ ]:
msno.dendrogram(df)
plt.show()

Columns joined at a **short distance** disappear as a team. The tighter the join, the more they behave like one shared decision — here, the privacy-guarded respondent.

## 🔍 Detective work: is `trust_news` MNAR?

We suspect people who distrust the news refuse to rate their trust. If so, the refusers should look different in the columns we **can** see. Let's check whether they differ in social media use:

In [ ]:
skipped = df["trust_news"].isna()

print("Refused the trust question:", skipped.sum(), "people\n")
for col in ["hours_social", "hours_news", "age"]:
    a = df.loc[skipped, col].mean()
    b = df.loc[~skipped, col].mean()
    print(f"{col:14s} refusers {a:5.2f} vs answered {b:5.2f}   (diff {a - b:+.2f})")

People who refused use **more social media** than those who answered. Small, but consistent — and remember from the EDA track that heavy social media use goes with *lower* trust. So the refusers are likely the low-trust group, exactly as suspected: **the answer itself caused the silence. That is MNAR.**

This is real detective work — with ordinary data this is as far as you can get, because the missing answers are gone forever.

## 🎭 The reveal

Our dataset is **synthetic**, which lets us do something no real survey can: look at the answers *before* anyone refused. Here is the truth the survey never saw.

In [ ]:
truth = pd.read_csv("data/eu_media_habits_answer_key.csv")
check = df.merge(truth, on="respondent_id")

# "deep distrust" = a trust rating of 4 or below
really_distrust = (check["trust_news_true"] <= 4).mean()
answered = check["trust_news"].notna()
you_would_report = (check.loc[answered, "trust_news"] <= 4).mean()

print(f"TRUE share who deeply distrust the news: {really_distrust:.1%}")
print(f"What your survey would report:           {you_would_report:.1%}")
print(f"\nYou would under-count media distrust by "
      f"{1 - you_would_report / really_distrust:.0%}.")

**Roughly one in three distrustful people is invisible in your results.**

Nothing was done wrong. No bug, no bad code. The survey ran correctly and still produced a conclusion that is meaningfully wrong — because the people least willing to answer were exactly the people the question was about.

If this were a real study on media trust in Europe, it would report a healthier picture than reality. Policy could be written on that number.

## ⚖️ Ethics lab — who disappears when you clean?

The standard reflex for missing data is `dropna()` — delete incomplete rows. **Beat 1 — Build it:** let's do exactly that. **Beat 2 — Measure it:** and count the cost.

In [ ]:
%run datalab_ethics.py

clean = df.dropna()
print(f"Before: {len(df)} respondents")
print(f"After dropna(): {len(clean)} respondents")
print(f"You deleted {len(df) - len(clean)} people — {1 - len(clean) / len(df):.0%} of your survey.\n")

representation(df, by="country")

**Beat 3 — Name it.** One line of 'cleaning' silently discarded about a third of the sample — and not a random third. The people deleted are disproportionately those who guard their privacy and those who distrust institutions. Their views leave the dataset entirely, and every number you publish afterwards is calculated *as if they did not exist*.

This connects straight to EURIDICE's commitments. Under the **GDPR**, consent must be *freely given* and can be withdrawn at any time (Art. 4(11), Art. 7) — refusing a question is a **right**, not a defect in your data. An analysis that silently deletes everyone who exercises that right turns a legal protection into a disadvantage. And where this data trains a high-risk system, the **EU AI Act** requires training data to be *relevant and sufficiently representative* (Art. 10(3)) — so quietly dropping a third of your respondents is a compliance problem as well as an ethical one.

The *Vienna Manifesto*'s call for inclusive technology begins with the same question: **whose reality does your dataset actually represent?**

**Beat 4 — Judge it.** Your call, in writing:

In [ ]:
my_decision = """
Your answer: you must publish a report on media trust from this survey.
Do you use dropna()? What do you tell readers about the people who refused?
"""

reflect_submit("missing-data-ethics", my_decision)

## Practical guidance

| Situation | Reasonable response |
|---|---|
| A few values, no pattern (MCAR) | Fill with median/mode, or drop — low risk |
| Gaps predicted by other columns (MAR) | Keep those columns and model it; note it |
| Gaps caused by the answer (MNAR) | **You cannot fix this with statistics.** Report it as a limitation |
| A whole column mostly empty | Consider dropping the *column*, not the people |

The professional habit is simple: **always look at missingness before you analyse, and always tell your reader what was missing.**

## ✅ What you learned

`missingno`'s matrix, bar, heatmap and dendrogram · MCAR vs MAR vs MNAR · how to gather evidence for MNAR from observed columns · that missing data can bias a headline finding by a third · why `dropna()` is an ethical decision, not a housekeeping one.

📚 **Go further:**
- [missingno on GitHub](https://github.com/ResidentMario/missingno) — every plot type
- [pandas: working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html) — the official guide

**Next:** Track 2 puts this to work — [Explore a real dataset].